# Understanding `atom`, `Bond`, and `molecule`

This notebook demonstrates how the three chemical objects work together.

- `atom` stores atomic identity and remaining valence.
- `Bond` stores two connected atoms and their bond order.
- `molecule` is a connected subgraph that records atoms and bonds and returns derived features.

The `molecule` class does not form or break bonds. The temporary `connect()` helper below represents work that will eventually belong to a `reaction_system` class.

In [14]:
from pathlib import Path
import sys

# Make the imports work whether the notebook starts in codeBase or src.
src_directory = Path.cwd()
if not (src_directory / "obj_node.py").exists():
    src_directory = src_directory / "src"
sys.path.insert(0, str(src_directory))

from obj_edge import Bond
from obj_node import atom
from obj_subgraph import molecule

## Example-only bond operation

`connect()` first checks both atoms, updates their remaining valence, and then creates a `Bond`. It lives outside `molecule`, preserving the separation between graph actions and subgraph data.

In [15]:
def connect(atom_i, atom_j, order=1):
    if not atom_i.has_sufficient_valence(order):
        raise ValueError(f"Atom {atom_i.index} has insufficient valence.")
    if not atom_j.has_sufficient_valence(order):
        raise ValueError(f"Atom {atom_j.index} has insufficient valence.")

    atom_i.form_bond(order)
    atom_j.form_bond(order)
    return Bond(atom_i, atom_j, order)


def show_features(species):
    return dict(zip(species.FEATURE_NAMES, species.features))

## Example 1: a single atom is a molecule

Every isolated atom is treated as a one-node connected subgraph. Oxygen begins with two units of remaining valence.

In [16]:
oxygen = atom(0, "O")
free_oxygen = molecule(atoms=[oxygen], bonds=[])

print("Element counts:", free_oxygen.element_counts)
print("Contains oxygen:", free_oxygen.has_atom(oxygen))
print("Connected:", free_oxygen.is_connected())
show_features(free_oxygen)

Element counts: {'C': 0, 'H': 0, 'O': 1}
Contains oxygen: True
Connected: True


{'n_C': 0.0,
 'n_H': 0.0,
 'n_O': 1.0,
 'total_remaining_valence': 2.0,
 'n_C-C': 0.0,
 'n_C=C': 0.0,
 'n_C#C': 0.0,
 'n_C-H': 0.0,
 'n_C-O': 0.0,
 'n_C=O': 0.0,
 'n_C#O': 0.0,
 'n_O-O': 0.0,
 'n_O=O': 0.0,
 'n_O-H': 0.0,
 'n_H-H': 0.0}

The first four global values are `n_C`, `n_H`, `n_O`, and `total_remaining_valence`. All bond counts are zero because the singleton has no edges.

## Example 2: water

Water contains one oxygen, two hydrogens, and two O–H single bonds. After both bonds form, every atom has zero remaining valence.

In [18]:
water_O = atom(1, "O")
water_H1 = atom(2, "H")
water_H2 = atom(3, "H")

oh_bond_1 = connect(water_O, water_H1, order=1)
oh_bond_2 = connect(water_O, water_H2, order=1)

water = molecule(
    atoms=[water_O, water_H1, water_H2],
    bonds=[oh_bond_1, oh_bond_2],
)

show_features(water)

{'n_C': 0.0,
 'n_H': 2.0,
 'n_O': 1.0,
 'total_remaining_valence': 0.0,
 'n_C-C': 0.0,
 'n_C=C': 0.0,
 'n_C#C': 0.0,
 'n_C-H': 0.0,
 'n_C-O': 0.0,
 'n_C=O': 0.0,
 'n_C#O': 0.0,
 'n_O-O': 0.0,
 'n_O=O': 0.0,
 'n_O-H': 2.0,
 'n_H-H': 0.0}

### Direct read-out of the atoms and bonds stored by `water`

A `molecule` stores its atom objects in `molecule.atoms` and its undirected chemical bond objects in `molecule.bonds`. Both containers are tuples. The following read-out verifies their contents, features, and object identities.

In [19]:
original_atom_names = ("water_O", "water_H1", "water_H2")
original_atoms = (water_O, water_H1, water_H2)
original_bond_names = ("oh_bond_1", "oh_bond_2")
original_bonds = (oh_bond_1, oh_bond_2)

print("Molecule class:", type(water).__name__)
print("water.atoms container type:", type(water.atoms).__name__)
print("number of stored atoms:", len(water.atoms))
print("\nATOM READ-OUT")
for position, (stored_atom, original_name, original_atom) in enumerate(
    zip(water.atoms, original_atom_names, original_atoms)
):
    print({
        "position_in_water.atoms": position,
        "python_class": type(stored_atom).__name__,
        "original_variable": original_name,
        "same_exact_object": stored_atom is original_atom,
        "atom_index": stored_atom.index,
        "element": stored_atom.element,
        "remaining_valence": stored_atom.remaining_valence,
        "features": stored_atom.features,
    })

print("\nwater.bonds container type:", type(water.bonds).__name__)
print("number of stored chemical bonds:", len(water.bonds))
print("\nBOND READ-OUT")
for position, (stored_bond, original_name, original_bond) in enumerate(
    zip(water.bonds, original_bond_names, original_bonds)
):
    print({
        "position_in_water.bonds": position,
        "python_class": type(stored_bond).__name__,
        "original_variable": original_name,
        "same_exact_object": stored_bond is original_bond,
        "node_indices": stored_bond.node_indices,
        "endpoint_elements": (
            stored_bond.node_i.element, stored_bond.node_j.element
        ),
        "order": stored_bond.order,
        "features": stored_bond.features,
        "endpoints_are_atoms_stored_by_water": (
            any(member is stored_bond.node_i for member in water.atoms)
            and any(member is stored_bond.node_j for member in water.atoms)
        ),
    })

assert water.atoms == original_atoms
assert water.bonds == original_bonds
assert all(isinstance(member, atom) for member in water.atoms)
assert all(isinstance(existing_bond, Bond) for existing_bond in water.bonds)
print("\nConfirmed: water stores all three atom objects and both Bond objects.")

Molecule class: molecule
water.atoms container type: tuple
number of stored atoms: 3

ATOM READ-OUT
{'position_in_water.atoms': 0, 'python_class': 'atom', 'original_variable': 'water_O', 'same_exact_object': True, 'atom_index': 1, 'element': 'O', 'remaining_valence': 0, 'features': [0.0, 0.0, 1.0, 0.0]}
{'position_in_water.atoms': 1, 'python_class': 'atom', 'original_variable': 'water_H1', 'same_exact_object': True, 'atom_index': 2, 'element': 'H', 'remaining_valence': 0, 'features': [0.0, 1.0, 0.0, 0.0]}
{'position_in_water.atoms': 2, 'python_class': 'atom', 'original_variable': 'water_H2', 'same_exact_object': True, 'atom_index': 3, 'element': 'H', 'remaining_valence': 0, 'features': [0.0, 1.0, 0.0, 0.0]}

water.bonds container type: tuple
number of stored chemical bonds: 2

BOND READ-OUT
{'position_in_water.bonds': 0, 'python_class': 'Bond', 'original_variable': 'oh_bond_1', 'same_exact_object': True, 'node_indices': (1, 2), 'endpoint_elements': ('O', 'H'), 'order': 1, 'features': [

The `same_exact_object` values confirm that `molecule` stores references to the original objects supplied to its constructor. It does not copy them. Each O–H bond is stored once as one undirected `Bond`; duplication into O→H and H→O happens only when `MoleculeEnv` builds `edge_index` for GNN message passing.

### What `has_atom()` checks

`has_atom()` checks object identity. A newly created oxygen with the same index is still a different Python object and is not part of `water`.

In [20]:
duplicate_O = atom(1, "O")

print("water_O has the same index as duplicate_O:", water_O.index == duplicate_O.index)
print("water_O is duplicate_O:", water_O is duplicate_O)
print("water contains water_O:", water.has_atom(water_O))
print("water contains duplicate_O:", water.has_atom(duplicate_O))

water_O has the same index as duplicate_O: True
water_O is duplicate_O: False
water contains water_O: True
water contains duplicate_O: False


### What `has_bond()` and `neighbors()` return

`has_bond()` returns the bond order (`1`, `2`, or `3`) and treats a bond as undirected. It returns `0` when no bond exists. `neighbors()` returns atoms exactly one bond away from the selected atom.

In [21]:
print("O-H1 bond order:", water.has_bond(water_O, water_H1))
print("H1-O bond order:", water.has_bond(water_H1, water_O))
print("H1-H2 bond order:", water.has_bond(water_H1, water_H2))
print("Neighbors of O:", [item.index for item in water.neighbors(water_O)])
print("Neighbors of H1:", [item.index for item in water.neighbors(water_H1)])

O-H1 bond order: 1
H1-O bond order: 1
H1-H2 bond order: 0
Neighbors of O: [2, 3]
Neighbors of H1: [1]


## Example 3: carbon dioxide and double-bond features

Carbon dioxide contains two C=O bonds. This example shows how `bond_type_counts` distinguishes double bonds from single bonds.

In [22]:
co2_C = atom(5, "C")
co2_O1 = atom(6, "O")
co2_O2 = atom(7, "O")

co2_bond_1 = connect(co2_C, co2_O1, order=2)
co2_bond_2 = connect(co2_C, co2_O2, order=2)

carbon_dioxide = molecule(
    atoms=[co2_C, co2_O1, co2_O2],
    bonds=[co2_bond_1, co2_bond_2],
)

print("Element counts:", carbon_dioxide.element_counts)
print("Number of C=O bonds:", carbon_dioxide.bond_type_counts[("C", "O", 2)])
print("C-O1 bond order:", carbon_dioxide.has_bond(co2_C, co2_O1))
show_features(carbon_dioxide)

Element counts: {'C': 1, 'H': 0, 'O': 2}
Number of C=O bonds: 2
C-O1 bond order: 2


{'n_C': 1.0,
 'n_H': 0.0,
 'n_O': 2.0,
 'total_remaining_valence': 0.0,
 'n_C-C': 0.0,
 'n_C=C': 0.0,
 'n_C#C': 0.0,
 'n_C-H': 0.0,
 'n_C-O': 0.0,
 'n_C=O': 2.0,
 'n_C#O': 0.0,
 'n_O-O': 0.0,
 'n_O=O': 0.0,
 'n_O-H': 0.0,
 'n_H-H': 0.0}

## Example 4: disconnected atoms are not one molecule

The constructor calls `_validate()` automatically. Two atoms without a connecting path must be represented as two singleton molecules, not one molecule.

In [13]:
separate_C = atom(8, "C")
separate_O = atom(9, "O")

try:
    molecule(atoms=[separate_C, separate_O], bonds=[])
except ValueError as error:
    print("Rejected as expected:", error)

carbon_species = molecule(atoms=[separate_C])
oxygen_species = molecule(atoms=[separate_O])
print("Singleton carbon connected:", carbon_species.is_connected())
print("Singleton oxygen connected:", oxygen_species.is_connected())

Rejected as expected: Atoms in a molecule must form one connected subgraph.
Singleton carbon connected: True
Singleton oxygen connected: True


## Summary

- `element_counts` and `bond_type_counts` summarize composition.
- `features` returns the fixed global subgraph features.
- `has_atom` checks membership, `has_bond` returns bond order or `0`, and `neighbors` returns local connectivity.
- `is_connected` verifies that all atoms form one component.
- `_validate` protects the molecule invariant during construction.
- Graph-edit actions remain outside `molecule`.